# Dataset playground

This notebook supports dataset edits and a small LoRA experiment. It compares responses after training on original and edited examples.

Run the edit cells first. Then run the training experiment below. The original dataset and production adapters remain unchanged.


In [1]:
import copy
import difflib
import json
from pathlib import Path
from datetime import datetime
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / "example_data").is_dir() and (ROOT.parent / "example_data").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "example_data").is_dir():
    raise FileNotFoundError("Open this notebook from the project directory.")

# Select a dataset. Lincoln examples also work with this notebook.
DATASET = "qianzhongshu_train.jsonl"
# DATASET = "lincoln_synthetic_modern.jsonl"

source_path = ROOT / "example_data" / DATASET
original = [json.loads(line) for line in source_path.read_text(encoding="utf-8").splitlines() if line.strip()]
working = copy.deepcopy(original)
print(f"Loaded {len(working)} examples from {source_path.name}")
print("The setup cell resets all in-memory edits each time it runs.")


Loaded 300 examples from qianzhongshu_train.jsonl
The setup cell resets all in-memory edits each time it runs.


## Select an example

Set `ROW` to an example index. The first example has index `0`.


In [ ]:
ROW = 0
example = copy.deepcopy(working[ROW])
print(json.dumps(example, ensure_ascii=False, indent=2))


## Edit the selected example

Uncomment an assignment below. Replace its text with your revision.

For Qian, use `instruction`, `input`, or `output`. For Lincoln, use the appropriate entry in `messages`.

Run this cell after each revision. To select another example, change `ROW` and rerun the selection cell first.


In [ ]:
# Qian examples:
# example["instruction"] = "请续写下面这段文字，保持钱钟书式的讽刺和机智。"
# example["input"] = "Your revised prompt"
# example["output"] = "Your revised target response"

# Lincoln examples: inspect the message roles in the selection cell first.
# example["messages"][-1]["content"] = "Your revised target response"
# example["historical_basis"] = "Your revised historical basis"

working[ROW] = copy.deepcopy(example)
print(f"Stored edits for example {ROW} in memory.")


## Compare all edits

Run this cell to compare the current examples with the original snapshot.

Removed lines have a `-` prefix. Added lines have a `+` prefix. Each heading identifies the example index.


In [ ]:
changed_rows = []
for index, (before, after) in enumerate(zip(original, working)):
    if before == after:
        continue
    changed_rows.append(index)
    display(Markdown(f"### Example {index}"))
    before_lines = json.dumps(before, ensure_ascii=False, indent=2, sort_keys=True).splitlines()
    after_lines = json.dumps(after, ensure_ascii=False, indent=2, sort_keys=True).splitlines()
    print("\n".join(difflib.unified_diff(
        before_lines, after_lines, fromfile="original", tofile="edited", lineterm=""
    )))
print(f"Changed examples: {len(changed_rows)} / {len(original)}")


## Undo the selected edit

Run this optional cell to restore the selected example from the snapshot.


In [ ]:
# working[ROW] = copy.deepcopy(original[ROW])
# example = copy.deepcopy(working[ROW])
# print(f"Restored example {ROW}.")


## Save an experimental copy

Run this cell only after review. The copy goes into `example_data/experiments/` with a timestamp.

The existing training scripts still use the canonical dataset. They do not automatically use this experimental copy.


In [ ]:
if working == original:
    print("No changes to save.")
else:
    output_dir = ROOT / "example_data" / "experiments"
    output_dir.mkdir(exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    output_path = output_dir / f"{source_path.stem}_{timestamp}.jsonl"
    with output_path.open("x", encoding="utf-8") as handle:
        for row in working:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Saved {len(working)} examples to {output_path}")
    print("The original dataset is unchanged.")


## Train a small A/B experiment

The experiment uses [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) and [PEFT LoRA](https://huggingface.co/docs/peft/quicktour).

It trains two fresh adapters with the same initialization seed, example order, and hyperparameters. Only the dataset differs.

This small model differs from the production 7B model. The experiment uses full-precision LoRA and supports Apple MPS, CUDA, or CPU. CPU execution can be slow.

The first model load requires internet access and disk space for the model cache. No cloud job or upload occurs.

1. Run the dependency cell once.
2. Restart the kernel if the installer requests it.
3. Run the setup and edit cells again after a restart.
4. Run the experiment cells in order.


In [ ]:
%pip install "torch>=2.2" "transformers>=4.45,<5" "peft>=0.13,<1" "accelerate>=0.34,<2"


### Configure the experiment

The subset includes every edited example plus unchanged examples up to `SUBSET_SIZE`. Each run starts from fresh model weights.

The loss applies only to target response tokens. This differs from the existing Qian trainer, which applies loss to the full sequence.

Long prompts keep their final `PROMPT_TOKENS` tokens. Long targets keep their first `TARGET_TOKENS` tokens. Edits outside these limits cannot affect training.


In [ ]:
import gc
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, get_peft_model

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
SEED = 2818
SUBSET_SIZE = 4
STEPS = 20
LEARNING_RATE = 2e-4
PROMPT_TOKENS = 256
TARGET_TOKENS = 128
NEW_TOKENS = 80
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

assert len(original) == len(working), "Keep the same row count for this edit experiment."
edited_indices = [i for i in range(len(original)) if original[i] != working[i]]
if not edited_indices:
    raise ValueError("Edit at least one example before this experiment.")
indices = edited_indices + [i for i in range(len(original)) if i not in edited_indices][:max(0, SUBSET_SIZE-len(edited_indices))]
experiment_original = copy.deepcopy([original[i] for i in indices])
experiment_edited = copy.deepcopy([working[i] for i in indices])
print(f"Device: {DEVICE}; examples per run: {len(indices)}; steps per run: {STEPS}")
print("Selected row indexes:", indices)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=False)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


In [ ]:
def prompt_and_target(row):
    if "messages" in row:
        messages = row["messages"]
        if not messages or messages[-1]["role"] != "assistant":
            raise ValueError("The final message must contain the assistant target.")
        return tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True), messages[-1]["content"]
    return (f"### 指令:\n{row['instruction']}\n\n### 输入:\n{row['input']}\n\n### 输出:\n", row["output"])


def encode_example(row):
    prompt, target = prompt_and_target(row)
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)[-PROMPT_TOKENS:]
    target_ids = tokenizer.encode(target, add_special_tokens=False)[:TARGET_TOKENS]
    if not prompt_ids or not target_ids:
        raise ValueError("Each example must contain prompt and target tokens.")
    # Add EOS only when the target is complete.
    if len(tokenizer.encode(target, add_special_tokens=False)) <= TARGET_TOKENS:
        target_ids.append(tokenizer.eos_token_id)
    return {"input_ids": torch.tensor([prompt_ids + target_ids], device=DEVICE),
            "labels": torch.tensor([[-100] * len(prompt_ids) + target_ids], device=DEVICE)}


def answer(model, prompt):
    model.eval()
    ids = tokenizer.encode(prompt, add_special_tokens=False)[-PROMPT_TOKENS:]
    inputs = torch.tensor([ids], device=DEVICE)
    with torch.inference_mode():
        output = model.generate(input_ids=inputs, attention_mask=torch.ones_like(inputs),
                                max_new_tokens=NEW_TOKENS, do_sample=False,
                                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0, len(ids):], skip_special_tokens=True)


def run_training(rows, probes, capture_base=False):
    set_seed(SEED)
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, token=False)
    model = get_peft_model(base, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0,
                                          target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))
    model.to(DEVICE)
    base_answers = [answer(model, prompt) for prompt in probes] if capture_base else None
    batches = [encode_example(row) for row in rows]
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
    order = list(range(len(batches)))
    random.Random(SEED).shuffle(order)
    losses = []
    model.train()
    for step in range(STEPS):
        optimizer.zero_grad(set_to_none=True)
        loss = model(**batches[order[step % len(order)]], use_cache=False).loss
        if not torch.isfinite(loss):
            raise ValueError("The training loss is not finite.")
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        losses.append(loss.item())
        if step == 0 or (step + 1) % 5 == 0:
            print(f"Step {step+1}/{STEPS}: loss={loss.item():.4f}")
    answers = [answer(model, prompt) for prompt in probes]
    del optimizer, model, base, batches, loss
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()
    return {"answers": answers, "losses": losses, "base_answers": base_answers}


### Choose fixed comparison prompts

The first probe uses the original prompt from an edited example. It measures adaptation to a familiar example.

The second probe is a new question. It provides a limited view of generalization. Replace it with your own question.

Both adapters receive exactly the same prompts. Greedy decoding reduces variation from random token selection.


In [ ]:
CUSTOM_QUESTION = "请谈谈现代人对手机的依赖。" if "messages" not in experiment_original[0] else "How can citizens disagree without hatred?"
custom = copy.deepcopy(experiment_original[0])
if "messages" in custom:
    custom["messages"] = [m for m in custom["messages"] if m["role"] == "system"] + [
        {"role": "user", "content": CUSTOM_QUESTION}, {"role": "assistant", "content": "placeholder"}]
else:
    custom["input"] = CUSTOM_QUESTION
probes = [prompt_and_target(experiment_original[0])[0], prompt_and_target(custom)[0]]
print("Probe 1: original prompt from edited example")
print("Probe 2:", CUSTOM_QUESTION)
# Verify that edits survive token truncation before either training run.
if all(torch.equal(encode_example(a)["input_ids"], encode_example(b)["input_ids"])
       for a, b in zip(experiment_original, experiment_edited)):
    raise ValueError("No token changes remain. Edit response text or increase token limits.")


### Run both training experiments

Run this cell to load the model and train both adapters. Progress appears every five steps.

A new run replaces the previous in-memory results. The experiment does not save adapter files or change the deployed Space.


In [ ]:
print("A: original dataset")
result_a = run_training(experiment_original, probes, capture_base=True)
print("B: edited dataset")
result_b = run_training(experiment_edited, probes)


### Compare model responses

The base column shows the response before adapter training. A and B show responses after the same number of training steps.

A different response demonstrates sensitivity to the edit. It does not prove higher quality. Identical responses can occur after small edits or short runs.

Training loss describes fit to each run's targets. Different targets make the two losses unsuitable as a direct quality ranking.


In [ ]:
from html import escape
from IPython.display import HTML

for i, prompt in enumerate(probes):
    display(Markdown(f"### Probe {i+1}"))
    print(prompt)
    texts = [result_a["base_answers"][i], result_a["answers"][i], result_b["answers"][i]]
    display(HTML("<table><tr><th>Base model</th><th>A: Original data</th><th>B: Edited data</th></tr><tr>" +
                 "".join("<td style='vertical-align:top;white-space:pre-wrap'>" + escape(text) + "</td>" for text in texts) + "</tr></table>"))
    print("A and B responses differ:", texts[1] != texts[2])


To try another edit, return to the selection and edit cells. Then rerun the experiment from its configuration cell.

Do not rerun the dataset setup cell unless you want to discard in-memory edits.
